# Grain-size statistics + core photos at real depth scale – events only

Built for your files:
* `GUAC_29A_GSD_Removedless1um_FolkWard_Stats.csv`, `GUAC_24A_GSD_Removedless1um_FolkWard_Stats_1.csv`
* `Lachua_Graine_Siza_Samples_N_Events_with_Ages.xlsx` (event depths, read as they are; "ABSENT" skipped)
* the two core photos (`GUAC-29A-1G-1-W.jpg`, `GUAC-24A-1G-1-W.jpg`)

Per core it makes:
1. **D10, D50, D90** in one figure (µm, log axis). D10 dotted with circles, D50
   solid with squares, D90 dashed with triangles.
2. **Sorting** (Folk & Ward 1957, phi, straight from your file).
3. The **core photo**, cropped to the depth window.
4. **COMBINED: photo | D10-D50-D90 | Sorting** on one depth axis.

**Where D10/D50/D90 come from.** Your stats files have no D10/D50/D90 columns,
only the Folk & Ward mean and sorting and the % in each Udden–Wentworth class.
D10/D50/D90 are read from those class percentages: the cumulative curve over the
1-phi classes, interpolated within each class (as GRADISTAT does). The clay class
runs to 1 µm because particles < 1 µm were removed (`CLAY_FINE_UM`). Cell 2
checks this by recomputing the Folk & Ward mean the same way and comparing it
with your `Mean_phi`. For your data they agree within 0.04 phi in every sample.

* **Real scale:** 1 cm of core = 1 cm on the page (`SCALE`). On every page the
  boxes start 2 cm below the top. Place the PDFs at 100 %.
* **Depth window:** 29A 15–88 cm, 24A 13–80 cm (`DEPTH_WINDOW`).
* **Photos:** 29A is turned 90° clockwise (it was scanned lying down). The
  depth at the image edges (`top_cm` / `bot_cm`) was measured from the ruler.
  The photo is never squeezed or stretched.
* **Same colour = same event number** in both cores. **Touching events** (E05 |
  E06 | E07, E15 | E16) get a dashed contact line.

In [ ]:
# =============================================================================
# CELL 1 - SETTINGS + UPLOAD
# Grain-size statistics (already calculated) plotted at REAL depth scale,
# event intervals only, ONE figure per variable per core, ready to be placed
# next to a core photo.
# =============================================================================
import pandas as pd, numpy as np, re, os, glob, zipfile
import matplotlib, matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, MaxNLocator
from google.colab import files

# ---- 1. your files (words that appear in the file names) --------------------
FILES = {'GUAC-29A': 'stats 29A',
         'GUAC-24A': 'stats 24A'}

# ---- 2. event depths ---------------------------------------------------------
# Your event depth file is read automatically (any uploaded csv/xlsx whose
# name contains EVENT_FILE). It overrides the lists below. The lists are only
# used if no event file is uploaded.
EVENT_FILE = 'event'      # word in the file name, e.g. 'Event_depths.xlsx'
EVENTS = {
 'GUAC-29A': {1:(13.40,14.10), 2:(15.10,19.10), 3:(22.60,30.15), 4:(31.30,33.20),
              5:(33.55,36.80), 6:(36.80,41.00), 7:(41.00,49.10), 8:(53.25,55.80),
              9:(56.80,58.10),10:(60.20,60.60),11:(61.20,64.10),12:(64.65,66.60),
             13:(67.15,67.50),14:(67.90,68.30),15:(70.80,71.15),16:(71.25,71.50),
             17:(73.15,75.40),18:(76.65,80.00),19:(82.45,87.75)},
 'GUAC-24A': {1:(11.75,11.90), 2:(12.90,17.40), 3:(19.35,26.20), 4:(27.50,27.85),
              5:(28.35,30.80), 6:(30.90,33.00), 7:(33.00,48.00), 8:(51.70,53.00),
             10:(55.70,55.85),11:(56.50,62.00),14:(63.50,63.75),15:(66.25,66.50),
             16:(66.70,67.10),17:(68.80,69.75),18:(70.15,70.55),19:(71.30,72.70),
             20:(75.20,79.80)},
}

# ---- 3. depth window of each figure = the part of the core in your photo ---
#         The plot's top edge is TOP cm and its bottom edge is BOTTOM cm.
DEPTH_WINDOW = {'GUAC-29A': (15.0, 88.0),
                'GUAC-24A': (13.0, 80.0)}

# ---- 3b. core photos -----------------------------------------------------------
# For each core: a word from the photo's file name, and the core DEPTH (cm) at
# the very TOP and very BOTTOM edge of that image. The photo is cropped to
# DEPTH_WINDOW and saved at the same real scale as the plots.
# left / right = keep only this part of the image width (0 = left edge,
# 1 = right edge), e.g. (0.2, 0.8) to cut off a ruler or label.
# rotate: 'cw' = turn the image 90° clockwise first (for a photo taken lying
# down with the core top on the LEFT, like the 29A scan). None = no turn.
# top_cm / bot_cm below were measured from the ruler in the photos you sent
# (10-cm colour blocks, straight to within 1.7 mm). They are depths at the
# image EDGES, so they hold at any resolution as long as the photo is not
# cropped differently.
PHOTOS = {
 'GUAC-29A': dict(file='29A', rotate='cw', top_cm=0.04, bot_cm=100.55,
                  left=0.0, right=1.0),
 'GUAC-24A': dict(file='24A', rotate=None, top_cm=-0.09, bot_cm=105.65,
                  left=0.0, right=1.0),
}

# ---- 4. real scale -----------------------------------------------------------
SCALE = 1.0          # cm on the page per cm of core. 1.0 = true size (1:1).
                     # 0.5 = half size. Use the SAME value as your photo.
PANEL_W_CM = 4.0     # width of the data box, cm
MARGIN_CM = dict(left=1.6, right=1.1, top=2.0, bottom=0.6)   # space for labels

# ---- 5. D10 / D50 / D90 -----------------------------------------------------------
# Your stats files hold Folk & Ward mean + sorting and the % in each
# Udden-Wentworth class, but no D10/D50/D90. These are then read from the
# class percentages: the cumulative curve over the 1-phi classes, interpolated
# within each class (as GRADISTAT does). Cell 2 checks the method by
# recomputing the Folk & Ward mean the same way and comparing it with your file.
CLAY_FINE_UM = 1.0   # finest size in the clay class: your files say "Removed
                     # less 1 um", so clay runs from 3.9 µm down to 1 µm

# ---- 6. columns in your stats file ---------------------------------------------
# Column names are found automatically. If one is wrong, write the exact name
# from your csv here, e.g. 'Mean': 'MEAN (φ)'.
COLUMNS = {'D10': None, 'D50': None, 'D90': None, 'Mean': None, 'Sorting': None}
# Units are read from the column name ('µm', 'phi', 'φ'). Nothing is converted.
# If a name has no unit, set it here: 'phi' or 'µm'.
UNITS   = {'D10': None, 'D50': None, 'D90': None, 'Mean': None, 'Sorting': 'phi'}
SHADE_EVENTS = True       # pale band behind each event interval
LABEL_EVENTS = True       # 'E03' next to each event

PALETTE = ['#1f77b4','#d62728','#2ca02c','#ff7f0e','#9467bd','#8c564b',
           '#e377c2','#17becf','#bcbd22','#7f3f00','#3f51b5','#00897b']

matplotlib.rcParams['pdf.fonttype'] = 42      # editable text in the PDF
matplotlib.rcParams['font.size'] = 8

print("Upload: the two grain-size stats csv files, the event depth xlsx, and the "
      "two core photos (jpg/png/tif).")
try: files.upload()
except Exception: pass
print("In /content:", sorted(os.path.basename(f) for f in glob.glob('/content/*')
                             if os.path.isfile(f)))

In [ ]:
# =============================================================================
# CELL 2 - READ THE FILES: depths, the five statistics, their units
# =============================================================================
def norm(x):
    return re.sub(r'[^a-z0-9]', '', str(x).lower())

def find_file(spec):
    """File whose name contains every word of spec (case, _ and - ignored)."""
    words = [norm(w) for w in re.split(r'[\s_\-]+', spec) if norm(w)]
    hits = [f for f in glob.glob('/content/*')
            if f.lower().endswith(('.csv', '.xlsx', '.xls'))
            and not os.path.basename(f).startswith(('GS_STATS_', 'EVENT_TABLE_current'))
            and all(w in norm(os.path.basename(f)) for w in words)]
    # (GS_STATS_ / EVENT_TABLE_ files are written by the big notebook - skipped)
    assert hits, f"no uploaded file name contains all of {words}"
    return max(hits, key=os.path.getmtime)               # newest upload

def read_table(path):
    return (pd.read_excel(path) if path.lower().endswith(('.xlsx', '.xls'))
            else pd.read_csv(path, encoding='utf-8-sig'))

INTERVAL = re.compile(r'(\d+\.?\d*)\s*(?:-|–|to)\s*(\d+\.?\d*)')

def sample_depth(label):
    """'GUAC_29A 15.5-16.0cm' -> (15.5, 16.0)     'GUAC-24A-1G-1-W 13.0cm' -> (13.0, 13.0)"""
    s = str(label)
    m = [(float(a), float(b)) for a, b in INTERVAL.findall(s) if float(a) < float(b) <= 500]
    if m: return m[-1]
    m = re.findall(r'(\d+\.?\d*)\s*cm', s, re.I) or re.findall(r'\d+\.\d+', s)
    return (float(m[-1]), float(m[-1])) if m else (np.nan, np.nan)

def get_depths(df):
    """Top and bottom of every sample, from depth columns or from the labels."""
    cols = {norm(c): c for c in df.columns}
    top = next((cols[k] for k in cols if k.startswith(('depthtop', 'top'))), None)
    bot = next((cols[k] for k in cols if k.startswith(('depthbot', 'depthbase', 'bottom', 'base'))), None)
    if top and bot:
        return pd.to_numeric(df[top], errors='coerce'), pd.to_numeric(df[bot], errors='coerce')
    dcol = next((cols[k] for k in cols if 'depth' in k), None)
    if dcol is not None and pd.to_numeric(df[dcol], errors='coerce').notna().mean() > 0.9:
        v = pd.to_numeric(df[dcol], errors='coerce'); return v, v
    for c in df.columns:                                  # sample labels
        if pd.api.types.is_numeric_dtype(df[c]): continue  # numbers are not labels
        d = [sample_depth(v) for v in df[c].head(20)]
        if np.mean([np.isfinite(a) for a, _ in d]) > 0.5:
            d = np.array([sample_depth(v) for v in df[c]])
            print(f"   depths read from the labels in column '{c}'")
            return pd.Series(d[:, 0]), pd.Series(d[:, 1])
    raise ValueError(f"no depth found in columns {list(df.columns)}")

# Column names as different programs write them (after removing spaces,
# brackets, dots and symbols):  D10 (µm) -> d10m,  Dx (10) -> dx10,
# Dv(10) -> dv10,  d(0.1) -> d01 (Malvern Mastersizer 2000),  D10 -> d10
PATTERNS = {'D10': ('d10', 'dx10', 'dv10', 'd01', 'dx01', 'dv01', 'p10'),
            'D50': ('d50', 'dx50', 'dv50', 'd05', 'dx05', 'dv05', 'p50', 'median'),
            'D90': ('d90', 'dx90', 'dv90', 'd09', 'dx09', 'dv09', 'p90'),
            'Mean': ('mean', 'mz'),
            'Sorting': ('sorting', 'sd', 'sigma')}

def to_number(series):
    """Numbers even when cells hold text: '12.3 µm', '12,3', ' 12.3 ' -> 12.3"""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    txt = series.astype(str).str.replace(',', '.', regex=False)
    return pd.to_numeric(txt.str.extract(r'([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)')[0],
                         errors='coerce')

def unit_of(colname):
    n = str(colname).lower()
    if 'phi' in n or 'φ' in n or '(f)' in n: return 'phi'
    if any(u in n for u in ('µm', 'μm', 'um', 'micron')): return 'µm'
    return None

def pick_column(df, var):
    """The column for one statistic. If the file has it twice (µm and phi):
    D10/D50/D90 -> the µm one;  Mean/Sorting -> the phi one (Folk & Ward)."""
    if COLUMNS.get(var): return COLUMNS[var]
    hits = [c for c in df.columns if norm(c).startswith(PATTERNS[var])]
    want = 'µm' if var.startswith('D') else 'phi'
    pref = [c for c in hits if unit_of(c) == want]
    return (pref or hits or [None])[0]

# Udden-Wentworth classes as phi intervals (coarse -> fine). The names are
# matched after removing spaces/dots/brackets, e.g. 'V.fine silt(7-8)'.
CLASSES = [('granule',    -2, -1), ('vcoarsesand', -1, 0), ('verycoarsesand', -1, 0),
           ('coarsesand',  0,  1), ('mediumsand',   1, 2), ('finesand',        2, 3),
           ('vfinesand',   3,  4), ('veryfinesand', 3, 4), ('coarsesilt',      4, 5),
           ('mediumsilt',  5,  6), ('finesilt',     6, 7), ('vfinesilt',       7, 8),
           ('veryfinesilt', 7, 8), ('clay',          8, None)]

def class_columns(df):
    """[(column, phi_coarse, phi_fine), ...] coarse -> fine, if the file has
    Udden-Wentworth class percentages."""
    found = {}
    for c in df.columns:
        n = norm(c)
        for key, lo, hi in CLASSES:
            if n.startswith(key) and (lo not in found):
                if hi is None: hi = -np.log2(CLAY_FINE_UM / 1000)   # 1 µm = 9.97 phi
                found[lo] = (c, lo, hi); break
    cols = [found[k] for k in sorted(found)]
    return cols if len(cols) >= 8 else []

def phi_at(row, edges, pct_coarser):
    """phi where the cumulative % (coarse -> fine) reaches pct_coarser."""
    row = np.nan_to_num(row)
    if row.sum() <= 0: return np.nan
    cum = np.concatenate([[0], np.cumsum(row) / row.sum() * 100])
    return np.interp(pct_coarser, cum, edges)

DATA, UNIT_OF = {}, {}
for core, spec in FILES.items():
    path = find_file(spec)
    print(f"\n{core}  <-  {os.path.basename(path)}")
    raw = read_table(path)
    top, bot = get_depths(raw)
    out = pd.DataFrame({'top_cm': top.values, 'bot_cm': bot.values})
    for var in PATTERNS:
        c = pick_column(raw, var)
        if c is None:
            if var.startswith('D') and class_columns(raw):
                continue      # calculated below from the Udden-Wentworth classes
            print(f"   !!! {var:8s} NOT FOUND. Columns in the file: {list(raw.columns)}")
            print(f"       -> if one of them is {var}, write its exact name in "
                  f"COLUMNS['{var}'] (Cell 1)")
            continue
        out[var] = to_number(raw[c]).values
        if out[var].notna().sum() == 0:
            print(f"   !!! {var}: column '{c}' has no numbers. First cells: "
                  f"{raw[c].head(3).tolist()}")
        u = UNITS.get(var) or unit_of(c)
        if u is None:                       # name has no unit
            if var == 'Mean' and 'D50' in out:
                # read both ways; the one closer to D50 (µm) is right
                m, d50 = out[var].median(), out['D50'].median()
                u = 'µm' if abs(np.log(m / d50)) < abs(np.log(1000 * 2**-m / d50)) else 'phi'
            else:
                u = 'phi' if var == 'Sorting' else 'µm'
        UNIT_OF[(core, var)] = u
        print(f"   {var:8s} <- column '{c}'   unit: {u}   "
              f"range {out[var].min():.3g}-{out[var].max():.3g}")
    # ---- D10/D50/D90 from the Udden-Wentworth class % if the file has none
    missing_d = [v for v in ('D10', 'D50', 'D90') if v not in out]
    cls = class_columns(raw)
    if missing_d and cls:
        P = np.column_stack([to_number(raw[c]).values for c, _, _ in cls])
        edges = [cls[0][1]] + [hi for _, _, hi in cls]           # phi, coarse -> fine
        pct = {p: np.array([phi_at(r, edges, p) for r in P]) for p in (5, 10, 16, 50, 84, 90, 95)}
        for v, p_coarser in (('D10', 90), ('D50', 50), ('D90', 10)):
            out[v] = 1000 * 2.0 ** -pct[p_coarser]                # phi -> µm
            UNIT_OF[(core, v)] = 'µm'
        print(f"   D10/D50/D90 not in the file -> read from the {len(cls)} "
              f"Udden-Wentworth class columns ({cls[0][0]} ... {cls[-1][0]})")
        # check: Folk & Ward mean and sorting the same way vs. your file
        mz = (pct[16] + pct[50] + pct[84]) / 3
        sg = (pct[84] - pct[16]) / 4 + (pct[95] - pct[5]) / 6.6
        if 'Mean' in out and UNIT_OF.get((core, 'Mean')) == 'phi':
            dm = np.abs(mz - out['Mean'].values)
            print(f"   check: Folk & Ward mean from the classes vs your Mean column: "
                  f"median diff {np.nanmedian(dm):.3f} phi, max {np.nanmax(dm):.3f} phi"
                  f"{'   OK' if np.nanmax(dm) < 0.1 else '   <-- LARGE, check CLAY_FINE_UM'}")
        if 'Sorting' in out:
            dsg = np.abs(sg - out['Sorting'].values)
            print(f"   check: sorting from the classes vs your Sorting column: "
                  f"median diff {np.nanmedian(dsg):.3f}, max {np.nanmax(dsg):.3f}")
    elif missing_d:
        print(f"   !!! no D10/D50/D90 columns and no class % columns found")
    out['mid_cm'] = (out.top_cm + out.bot_cm) / 2
    out = out.dropna(subset=['mid_cm'])
    # repeated measurements of one sample -> averaged
    out = out.groupby(['top_cm', 'bot_cm'], as_index=False).mean().sort_values('mid_cm')
    DATA[core] = out.reset_index(drop=True)
    print(f"   {len(out)} samples, {out.top_cm.min():.2f}-{out.bot_cm.max():.2f} cm")
    print(f"   first rows - compare with your file:")
    print(out.head(3).round(3).to_string(index=False))

# ---- event depths from your event file ----------------------------------------
def event_number(v):
    m = re.search(r'(\d+)', str(v))
    return int(m.group(1)) if m else None

def read_event_file(path):
    """Two layouts are understood:
    WIDE - one block of columns per core:
        age_CE | age_unc | event | in_24A | 24A_top_cm | 24A_base_cm | ... |
                                   in_29A | 29A_top_cm | 29A_base_cm | ...
    LONG - one row per event per core:
        core | event | top_cm | base_cm
    'absent' / 'no' in an in_<core> column, or an empty depth, = not in that core."""
    t = read_table(path)
    t.columns = [str(c).strip() for c in t.columns]
    ev = {'GUAC-29A': {}, 'GUAC-24A': {}}
    ecol = next((c for c in t.columns if norm(c) in ('event', 'eventid', 'eventno')), None)
    assert ecol, f"no 'event' column in {list(t.columns)}"
    blocks = {}
    for c in t.columns:                                   # WIDE layout?
        n = norm(c)
        for tag, core in (('24a', 'GUAC-24A'), ('29a', 'GUAC-29A')):
            if n.startswith(tag) or n.startswith('in' + tag):
                b = blocks.setdefault(core, {})
                if n.startswith('in'):                  b['in'] = c
                elif n.endswith(('topcm', 'top')):     b['top'] = c
                elif n.endswith(('basecm', 'base', 'bottomcm', 'bottom')): b['base'] = c
    blocks = {k: v for k, v in blocks.items() if 'top' in v and 'base' in v}
    if blocks:
        for _, r in t.iterrows():
            e = event_number(r[ecol])
            if e is None: continue
            for core, b in blocks.items():
                flag = str(r[b['in']]).strip().lower() if 'in' in b else 'yes'
                top = pd.to_numeric(r[b['top']], errors='coerce')
                base = pd.to_numeric(r[b['base']], errors='coerce')
                if flag.startswith(('absent', 'no', 'n/a')) or not (
                        np.isfinite(top) and np.isfinite(base)):
                    continue
                ev[core][e] = (float(min(top, base)), float(max(top, base)))
        layout = 'wide (one column block per core)'
    else:                                                  # LONG layout
        ccol = next((c for c in t.columns if norm(c).startswith('core')), None)
        tcol = next((c for c in t.columns if norm(c).startswith(('top', 'depthtop'))), None)
        bcol = next((c for c in t.columns if norm(c).startswith(('base', 'bottom', 'depthbase', 'depthbot'))), None)
        assert ccol and tcol and bcol, f"cannot find core / top / base columns in {list(t.columns)}"
        for _, r in t.iterrows():
            e = event_number(r[ecol]); cn = norm(r[ccol])
            core = 'GUAC-29A' if '29a' in cn else 'GUAC-24A' if '24a' in cn else None
            top = pd.to_numeric(r[tcol], errors='coerce'); base = pd.to_numeric(r[bcol], errors='coerce')
            if e is None or core is None or not (np.isfinite(top) and np.isfinite(base)): continue
            ev[core][e] = (float(min(top, base)), float(max(top, base)))
        layout = 'long (one row per event)'
    return ev, layout

ev_files = [f for f in glob.glob('/content/*')
            if f.lower().endswith(('.csv', '.xlsx', '.xls'))
            and norm(EVENT_FILE) in norm(os.path.basename(f))
            and not os.path.basename(f).startswith('EVENT_TABLE_current')]
if ev_files:
    path = max(ev_files, key=os.path.getmtime)
    EVENTS, layout = read_event_file(path)
    print(f"\nEvent depths <- {os.path.basename(path)}  ({layout})")
else:
    print(f"\n!!! No event file (name containing '{EVENT_FILE}') uploaded - "
          f"using the lists in Cell 1. Upload your event depth file and re-run.")
for core in EVENTS:
    print(f"   {core}: {len(EVENTS[core])} events")
    for e, (t, b) in sorted(EVENTS[core].items()):
        print(f"      E{e:02d}  {t:6.2f} - {b:6.2f} cm")

# one colour per event NUMBER, the same in both cores
ALL_IDS = sorted({e for c in EVENTS for e in EVENTS[c]})
EV_COLOR = {e: PALETTE[i % len(PALETTE)] for i, e in enumerate(ALL_IDS)}

In [ ]:
# =============================================================================
# CELL 3 - PLOT at real depth scale, events only
#   per core:  (1) D10-D50-D90 figure   (2) Sorting figure   (3) photo figure
#              (4) COMBINED: photo | D10-D50-D90 | Sorting on one depth axis
#
# Real scale: every data / photo box is exactly (bottom - top) x SCALE cm tall
# and starts MARGIN_CM['top'] cm below the page top. Figures are saved WITHOUT
# 'tight' cropping, so the PDF keeps that size. Place at 100 %.
# The photo keeps its true proportions: its width on the page comes from the
# image itself (same cm per pixel across as down), so it is never squeezed
# or stretched.
# =============================================================================
from PIL import Image
from matplotlib.ticker import LogLocator, FuncFormatter, NullFormatter
from matplotlib.lines import Line2D
from IPython.display import Image as ShowImage, display
Image.MAX_IMAGE_PIXELS = None                    # core scans can be very large

CM = 1 / 2.54                                    # inches per cm
OUT = '/content/realscale'
os.makedirs(OUT, exist_ok=True)

D_AXIS = 'log'     # 'log' = D10, D50, D90 all readable (D90 is ~10x D10)
                   # 'linear' = plain µm axis
D_STYLE = {'D10': dict(marker='o', ls=':',  ms=2.6),
           'D50': dict(marker='s', ls='-',  ms=2.6),
           'D90': dict(marker='^', ls='--', ms=3.0)}
SORT_W_CM = 3.0    # width of the Sorting panel, cm (D panel = PANEL_W_CM)
GAP_CM    = 0.5    # space between panels in the combined figure, cm
CONTACT_TOL = 0.15 # cm: events closer than this are "touching"
CONTACT_KW  = dict(color='0.25', lw=0.8, ls=(0, (4, 2)), zorder=3)
PHOTO_EVENT_MARKS = True   # coloured bar beside the photo for each event
D_VARS = ['D10', 'D50', 'D90']

# ---- helpers -------------------------------------------------------------------
def event_samples(df, var, top, base):
    """Samples whose MIDDLE lies inside the event, with a value for var."""
    m = (df.mid_cm >= top) & (df.mid_cm <= base) & df[var].notna()
    return df[m]

def visible_events(core):
    t0, t1 = DEPTH_WINDOW[core]
    return [(e, t, b) for e, (t, b) in sorted(EVENTS[core].items(), key=lambda x: x[1])
            if b >= t0 and t <= t1]

def contacts(core):
    """Depths where one event ends and the next starts (E05/E06/E07 ...)."""
    ev = visible_events(core)
    return [(ev[i][2] + ev[i + 1][1]) / 2 for i in range(len(ev) - 1)
            if ev[i + 1][1] - ev[i][2] <= CONTACT_TOL]

def depth_axis(ax, core, labels=True):
    d_top, d_bot = DEPTH_WINDOW[core]
    ax.set_ylim(d_bot, d_top)
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_minor_locator(MultipleLocator(1))
    if labels: ax.set_ylabel('Depth (cm)')
    else: ax.tick_params(axis='y', which='both', labelleft=False)

def event_labels(ax, core):
    d_top, d_bot = DEPTH_WINDOW[core]
    for e, t, b in visible_events(core):
        ax.text(1.03, (max(t, d_top) + min(b, d_bot)) / 2, f'E{e:02d}',
                color=EV_COLOR[e], fontsize=6.5, fontweight='bold', va='center',
                ha='left', transform=ax.get_yaxis_transform(), clip_on=False)

def unit_text(unit):
    return 'phi' if unit == 'phi' else unit           # plain text only

# ---- the two data panels ------------------------------------------------------
def draw_data(ax, core, vars_):
    """Event bands, contact lines and the curves of vars_ in one box."""
    df = DATA[core]
    vars_ = [v for v in vars_ if v in df and df[v].notna().any()]
    vals = []
    for e, t, b in visible_events(core):
        col = EV_COLOR[e]
        if SHADE_EVENTS:
            ax.axhspan(t, b, color=col, alpha=0.10, lw=0, zorder=0)
        for var in vars_:
            seg = event_samples(df, var, t, b)
            if seg.empty: continue
            vals += list(seg[var])
            st = D_STYLE.get(var, dict(marker='o', ls='-', ms=2.6))
            ax.plot(seg[var], seg.mid_cm, color=col, lw=0.9, ls=st['ls'],
                    marker=st['marker'], ms=st['ms'], markeredgewidth=0, zorder=2)
    for y in contacts(core):
        ax.axhline(y, **CONTACT_KW)
    is_d = bool(vars_) and vars_[0].startswith('D')
    if vals:
        lo, hi = np.nanmin(vals), np.nanmax(vals)
        if is_d and D_AXIS == 'log' and lo > 0:
            ax.set_xscale('log')
            ax.set_xlim(lo / 1.15, hi * 1.15)
            decades = np.log10(hi / lo)
            subs = (1, 2, 5) if decades < 1.6 else (1, 5) if decades < 2.2 else (1,)
            ax.xaxis.set_major_locator(LogLocator(base=10, subs=subs))  # fewer
            ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10)))
            ax.tick_params(axis='x', which='minor', length=2)          # labels
            ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
            ax.xaxis.set_minor_formatter(NullFormatter())
        else:
            pad = 0.06 * (hi - lo or 1)
            ax.set_xlim(lo - pad, hi + pad)
            ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.xaxis.tick_top(); ax.xaxis.set_label_position('top')
    for sp in ('right', 'bottom'): ax.spines[sp].set_visible(False)
    if not vars_:
        ax.text(0.5, 0.5, 'no data', transform=ax.transAxes, ha='center', color='r')
        return 0
    unit = UNIT_OF[(core, vars_[0])]
    if is_d:
        ax.set_xlabel(f"D10, D50, D90 ({unit_text(unit)})")
        key = [Line2D([], [], color='0.3', lw=0.9, ms=3, marker=D_STYLE[v]['marker'],
                      ls=D_STYLE[v]['ls'], label=v) for v in vars_]
        box_h_cm = (DEPTH_WINDOW[core][1] - DEPTH_WINDOW[core][0]) * SCALE
        ax.legend(handles=key, loc='lower center', ncol=len(vars_), fontsize=6.5,
                  bbox_to_anchor=(0.5, 1.0 + 0.9 / box_h_cm), frameon=False,
                  handlelength=2.2, columnspacing=0.8, borderaxespad=0)
    else:
        ax.set_xlabel(f"{vars_[0]} ({unit_text(unit)})")
    return len(vals)

# ---- the photo -------------------------------------------------------------------
def find_photo(spec):
    words = [norm(w) for w in re.split(r'[\s_\-]+', spec) if norm(w)]
    made_here = ('fig', 'evpca', 'pca', 'realscale', 'combined', 'preview')
    hits = [f for f in glob.glob('/content/*')
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff'))
            and all(w in norm(os.path.basename(f)) for w in words)
            and not any(m in norm(os.path.basename(f)) for m in made_here)]
    return max(hits, key=os.path.getmtime) if hits else None

def photo_crop(core):
    """Crop the photo to DEPTH_WINDOW. Returns the image, the depths it covers
    and its TRUE width on the page in cm (no squeezing or stretching)."""
    cfg = PHOTOS.get(core, {})
    path = find_photo(cfg.get('file', ''))
    if not path: return None
    img = Image.open(path).convert('RGB')
    if cfg.get('rotate') == 'cw':   img = img.transpose(Image.ROTATE_270)   # 90° clockwise
    elif cfg.get('rotate') == 'ccw': img = img.transpose(Image.ROTATE_90)
    w_px, h_px = img.size
    p_top, p_bot = cfg['top_cm'], cfg['bot_cm']            # depth at image edges
    d_top, d_bot = DEPTH_WINDOW[core]
    px_per_cm = h_px / (p_bot - p_top)
    r0, r1 = (d_top - p_top) * px_per_cm, (d_bot - p_top) * px_per_cm
    if r0 < 0 or r1 > h_px:
        print(f"   !!! {core}: the photo covers {p_top}-{p_bot} cm but the window is "
              f"{d_top}-{d_bot} cm - the missing part is left white")
    c0, c1 = int(cfg.get('left', 0) * w_px), int(cfg.get('right', 1) * w_px)
    crop = img.crop((c0, int(round(max(r0, 0))), c1, int(round(min(r1, h_px)))))
    width_cm = (c1 - c0) / px_per_cm * SCALE               # true proportions
    return dict(img=np.asarray(crop), path=path, px_per_cm=px_per_cm,
                top=p_top + max(r0, 0) / px_per_cm,
                bot=p_top + min(r1, h_px) / px_per_cm, width_cm=width_cm)

def draw_photo(ax, core, ph):
    ax.imshow(ph['img'], extent=[0, 1, ph['bot'], ph['top']], aspect='auto',
              interpolation='lanczos')
    ax.set_xlim(0, 1); ax.set_xticks([])
    # event bars sit in the gap just RIGHT of the photo, so no part of the
    # photo is covered; contacts get a short black tick on the same bar
    xb = 1 + 0.15 / ph['width_cm']                         # 1.5 mm right of photo
    d_top, d_bot = DEPTH_WINDOW[core]
    if PHOTO_EVENT_MARKS:
        for e, t, b in visible_events(core):
            ax.plot([xb, xb], [max(t, d_top), min(b, d_bot)], color=EV_COLOR[e],
                    lw=3, solid_capstyle='butt', clip_on=False, zorder=4)
    for y in contacts(core):
        ax.plot([xb - 0.08 / ph['width_cm'], xb + 0.08 / ph['width_cm']], [y, y],
                color='k', lw=1.0, clip_on=False, zorder=5)

# ---- one page with any row of panels, at exact cm --------------------------------
def page(core, panels, fname):
    """panels = list of (kind, width_cm, arg) with kind 'photo' | 'data'."""
    d_top, d_bot = DEPTH_WINDOW[core]
    box_h = (d_bot - d_top) * SCALE
    W = (MARGIN_CM['left'] + sum(w for _, w, _ in panels) +
         GAP_CM * (len(panels) - 1) + MARGIN_CM['right'])
    H = MARGIN_CM['top'] + box_h + MARGIN_CM['bottom']
    fig = plt.figure(figsize=(W * CM, H * CM))
    x, axes = MARGIN_CM['left'], []
    for i, (kind, w, arg) in enumerate(panels):
        ax = fig.add_axes([x / W, MARGIN_CM['bottom'] / H, w / W, box_h / H])
        if kind == 'photo': draw_photo(ax, core, arg)
        else: draw_data(ax, core, arg)
        depth_axis(ax, core, labels=(i == 0))
        axes.append(ax); x += w + GAP_CM
    event_labels(axes[-1], core)
    fig.text(0.25 / W, 1 - 0.25 / H, core, fontsize=8, fontweight='bold',
             ha='left', va='top')
    name = f"{OUT}/{core}_{fname}_realscale_{SCALE:g}x"
    fig.savefig(name + '.pdf', facecolor='white')            # NO bbox='tight'
    fig.savefig(name + '.png', dpi=150, facecolor='white')
    plt.close(fig)
    print(f"   {os.path.basename(name)}.pdf   page {W:.1f} x {H:.1f} cm, "
          f"boxes {box_h:.1f} cm tall, top {MARGIN_CM['top']:g} cm below page top")
    return name

# ---- make everything ---------------------------------------------------------------
for f_ in glob.glob(f'{OUT}/*'):                 # remove figures of earlier runs
    os.remove(f_)
COMBINED = {}
for core in DATA:
    df = DATA[core]
    print(f"\n{core}: depth {DEPTH_WINDOW[core][0]}-{DEPTH_WINDOW[core][1]} cm "
          f"at {SCALE:g}:1   contacts at {[round(y, 2) for y in contacts(core)]} cm")
    n_d = sum(event_samples(df, v, t, b).shape[0] for v in D_VARS if v in df
              for _, t, b in visible_events(core))
    n_s = sum(event_samples(df, 'Sorting', t, b).shape[0] for _, t, b in
              visible_events(core)) if 'Sorting' in df else 0
    print(f"   points plotted: D10/D50/D90 = {n_d}, Sorting = {n_s}")
    if n_d == 0:
        print(f"   !!! NO D-values plotted for {core}. Check the column lines in "
              f"Cell 2's printout.")
    page(core, [('data', PANEL_W_CM, D_VARS)], 'D10-D50-D90')
    page(core, [('data', SORT_W_CM, ['Sorting'])], 'Sorting')
    ph = photo_crop(core)
    if ph is None:
        print(f"   no photo found with '{PHOTOS.get(core, {}).get('file')}' in its name")
        COMBINED[core] = page(core, [('data', PANEL_W_CM, D_VARS),
                                     ('data', SORT_W_CM, ['Sorting'])], 'COMBINED')
        continue
    print(f"   photo {os.path.basename(ph['path'])}: {ph['px_per_cm']:.1f} px/cm, "
          f"true width {ph['width_cm']:.2f} cm on the page")
    page(core, [('photo', ph['width_cm'], ph)], 'Photo')
    COMBINED[core] = page(core, [('photo', ph['width_cm'], ph),
                                 ('data', PANEL_W_CM, D_VARS),
                                 ('data', SORT_W_CM, ['Sorting'])], 'COMBINED')

# preview: the combined figures themselves, shown small
for core, name in COMBINED.items():
    print(f"\n{core} - combined figure (shown small here; the PDF is real scale)")
    display(ShowImage(name + '.png', width=420))

In [ ]:
# =============================================================================
# CELL 4 - DOWNLOAD all real-scale figures (PDF + PNG) in one zip
# =============================================================================
ZIP = '/content/GrainSize_realscale.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f'{OUT}/*')):
        z.write(f, os.path.basename(f))
print(f"{ZIP}: {len(glob.glob(OUT + '/*'))} files")
files.download(ZIP)